In [64]:
import importlib
import importlib.util
import json
import os
from pathlib import Path
import sys
from dataclasses import dataclass, asdict

import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text
import tiktoken

# Local imports
import src.db_con as db_con
importlib.reload(db_con)
from src.db_con import get_config, list_models, get_session_training_progress, load_model_spec, get_top_session_model, get_session_models

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "apis").exists():
    REPO_ROOT = REPO_ROOT.parent

BENCHMARKING_DIR = REPO_ROOT / "benchmarking"
TRAINER_APP_DIR = REPO_ROOT / "apis" / "model-trainer" / "app"
for path in [BENCHMARKING_DIR, TRAINER_APP_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

In [2]:
# Load model-db config
get_config.cache_clear()
config = get_config()
print(f"DB password loaded: {bool(config.db_password)}")

DB password loaded: True


In [3]:
SETTINGS_PATH = REPO_ROOT / "benchmarking" / "training_params" / "settings_nas_bench_201.py"
if not SETTINGS_PATH.exists():
    SETTINGS_PATH = Path.cwd() / "settings_nas_bench_201.py"

settings_spec = importlib.util.spec_from_file_location("settings_nas_bench_201", SETTINGS_PATH)
nas_settings = importlib.util.module_from_spec(settings_spec)
settings_spec.loader.exec_module(nas_settings)

print(f"Loaded NAS-Bench-201 settings from {SETTINGS_PATH}")

Loaded NAS-Bench-201 settings from c:\Users\linki\Documents\AutoML\benchmarking\training_params\settings_nas_bench_201.py


### Training parameter definition
By model_ID

In [ ]:
run_a = "c1e5308eef93ac7009d74ea2df6ab9a6b69287468d5456eb7aebd4cef229965c"

In [ ]:
run_b = "1ad5989964f7a63d5f0565ab4eef9d999551e27fabf9742043d10423f895bf35"

In [4]:
run_c = "16368e2889ea1d11e0bf89018bbdcd022ca5da018e647f40d823379ce0907fdf"

In [5]:
# Change target ID here to select which benchmarking session to use for the rest of the notebook.
benchmarking_session_id = run_c

In [6]:
print(get_top_session_model(benchmarking_session_id))

{'model_id': 'e99bb7ce-e250-4c2f-bdbf-22ee8ff48aa7', 'test_accuracy_raw': 86.26, 'test_loss_raw': 1.5957778930664062}


In [7]:
# Set one of these values before running this cell.
selected_model_id = get_top_session_model(benchmarking_session_id)["model_id"]

model_info, model_spec = load_model_spec(selected_model_id, config)

print("Selected model:", model_info)
print("Model spec keys:", list(model_spec.keys()))

Selected model: {'id': 'e99bb7ce-e250-4c2f-bdbf-22ee8ff48aa7', 'name': 'WideResidualBNConvNet_32x32', 'created_at': '2026-07-21 12:32:42.337341+00:00', 'extra_meta': {'builder_notes': 'Expanded residual CNN with wider feature stages, affine BatchNorm2d layers, identity skip pathways, progressive channel scaling, and dropout-regularized classifier head to improve convergence speed and representational capacity on 32x32 image classification.', 'selection_mode': 'selected'}}
Model spec keys: ['edges', 'nodes', 'input_shape', 'exit_node_id', 'entry_node_id']


In [8]:
print(json.dumps(model_spec, indent=2))

{
  "edges": [
    {
      "source": "conv1",
      "target": "bn1"
    },
    {
      "source": "bn1",
      "target": "relu1"
    },
    {
      "source": "relu1",
      "target": "conv2"
    },
    {
      "source": "relu1",
      "target": "conv3",
      "edge_type": "skip"
    },
    {
      "source": "conv2",
      "target": "bn2"
    },
    {
      "source": "bn2",
      "target": "relu2"
    },
    {
      "source": "relu2",
      "target": "conv3"
    },
    {
      "source": "conv3",
      "target": "bn3"
    },
    {
      "source": "bn3",
      "target": "relu3"
    },
    {
      "source": "relu3",
      "target": "conv4"
    },
    {
      "source": "relu3",
      "target": "conv5",
      "edge_type": "skip"
    },
    {
      "source": "conv4",
      "target": "bn4"
    },
    {
      "source": "bn4",
      "target": "relu4"
    },
    {
      "source": "relu4",
      "target": "conv5"
    },
    {
      "source": "conv5",
      "target": "bn5"
    },
    {
      "source

In [9]:
# Define training parameters from the loaded NAS-Bench-201 settings
TRAINING_PARAMETERS = {
    "loss_function": "CrossEntropyLoss",
    "optimizer": nas_settings.optimizer,
    "epochs": 200,
    "dataset_name": "cifar10",  # Change to "cifar10", "cifar100" or "ImageNet16" as needed
    "batch_size": nas_settings.batch_size,
    "lr": nas_settings.initial_LR,
    "momentum": nas_settings.momentum,
    "weight_decay": nas_settings.weight_decay,
    "nesterov": nas_settings.Nesterov,
    "ending_lr": nas_settings.ending_LR,
    "lr_schedule": nas_settings.LR_schedule,
}

TRAIN_VIA_API = True
POLL_TIMEOUT_SECONDS = 7200
POLL_INTERVAL_SECONDS = 60*5

TRAIN_REQUEST = {
    "model_spec": model_spec,
    "training_parameters": TRAINING_PARAMETERS,
}

print(json.dumps(TRAINING_PARAMETERS, indent=2))

{
  "loss_function": "CrossEntropyLoss",
  "optimizer": "SGD",
  "epochs": 200,
  "dataset_name": "cifar10",
  "batch_size": 256,
  "lr": 0.1,
  "momentum": 0.9,
  "weight_decay": 0.0005,
  "nesterov": true,
  "ending_lr": 0,
  "lr_schedule": "cosine"
}


### API Training

In [ ]:
from src.train_api import get_job_result, get_job_status, submit_train_job, wait_for_job


if TRAIN_VIA_API:
    job = submit_train_job(TRAIN_REQUEST)
    print("Submitted training job:", job)

    final_status = wait_for_job(
        job["job_id"],
        timeout_seconds=POLL_TIMEOUT_SECONDS,
        poll_interval_seconds=POLL_INTERVAL_SECONDS,
    )
    print("Final status:", final_status)

    job_result = get_job_result(job["job_id"])
    print("Result:")
    print(json.dumps(job_result, indent=2))
else:
    print("TRAIN_VIA_API is False. Run the direct fallback cell next.")

### NATS bench benchmark (local)
The NATS Bench setup uses a fixed train/validation/test split for fair comparison. In this section, the model is still created from your saved model spec, but the data pipeline uses a CIFAR-10 split with 25K training images, 25K validation images, and the standard 10K test set.

In [38]:
# Functions from model-trainer
import torch
import nn_core
import random
import copy
import math
import time

from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10
from torchvision import transforms

def _split_cifar10_train_val_indices(dataset, seed=42):
    if not hasattr(dataset, "targets"):
        raise ValueError("The CIFAR-10 dataset does not expose targets needed for a balanced split.")

    class_to_indices = {class_id: [] for class_id in range(10)}
    for idx, target in enumerate(dataset.targets):
        class_to_indices[int(target)].append(idx)

    rng = random.Random(seed)
    train_indices = []
    val_indices = []

    for class_id in range(10):
        indices = class_to_indices[class_id]
        if len(indices) < 2:
            raise ValueError(f"Not enough samples for class {class_id} to create a split.")
        rng.shuffle(indices)
        split_point = len(indices) // 2
        train_indices.extend(indices[:split_point])
        val_indices.extend(indices[split_point:])

    return train_indices, val_indices


def get_cifar10_nats_bench_dataloaders(train_batch_size, val_batch_size, test_batch_size, num_workers, datadir='_dataset', resize=None, seed=42):
    """Create CIFAR-10 loaders using a NATS Bench-style 25K/25K train/validation split."""
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    size, pad = 32, 4

    if resize is None:
        resize = size

    train_transform = transforms.Compose([
        transforms.RandomCrop(size, padding=pad),
        transforms.Resize(resize),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    test_transform = transforms.Compose([
        transforms.Resize(resize),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_dataset = CIFAR10(datadir, True, train_transform, download=True)
    test_dataset = CIFAR10(datadir, False, test_transform, download=True)

    train_indices, val_indices = _split_cifar10_train_val_indices(train_dataset, seed=seed)
    train_subset = Subset(train_dataset, train_indices)
    val_subset = Subset(train_dataset, val_indices)

    train_loader = DataLoader(
        train_subset,
        train_batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_subset,
        val_batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset,
        test_batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader

In [14]:
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = str(REPO_ROOT / "datasets")

train_loader, val_loader, test_loader = get_cifar10_nats_bench_dataloaders(
    train_batch_size=256,
    val_batch_size=256,
    test_batch_size=256,
    num_workers=0,
    datadir=data_dir,
    seed=42,
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

model = nn_core.build_model_from_dict_via_pydantic(model_spec).to(torch_device)
criterion = torch.nn.CrossEntropyLoss()

base_lr = TRAINING_PARAMETERS["lr"]
momentum = TRAINING_PARAMETERS["momentum"]
weight_decay = TRAINING_PARAMETERS["weight_decay"]
num_epochs = TRAINING_PARAMETERS["epochs"]

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=base_lr,
    momentum=momentum,
    weight_decay=weight_decay,
    nesterov=True,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=0.0,
)

def evaluate(loader):
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(torch_device, non_blocking=True)
            targets = targets.to(torch_device, non_blocking=True)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            batch_size = targets.size(0)
            loss_sum += loss.item() * batch_size
            correct += (outputs.argmax(dim=1) == targets).sum().item()
            total += batch_size

    return {
        "acc": correct / total,
        "loss": loss_sum / total,
    }

best_val_acc = -1.0
best_epoch = -1
best_state_dict = None

for epoch in range(num_epochs):
    epoch_start = time.time()
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for inputs, targets in train_loader:
        inputs = inputs.to(torch_device, non_blocking=True)
        targets = targets.to(torch_device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)
        train_loss_sum += loss.item() * batch_size
        train_correct += (outputs.argmax(dim=1) == targets).sum().item()
        train_total += batch_size

    train_acc = train_correct / train_total
    train_loss = train_loss_sum / train_total

    val_metrics = evaluate(val_loader)

    current_lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - epoch_start

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        best_epoch = epoch + 1
        best_state_dict = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:03d}/{num_epochs:03d} | "
        f"lr={current_lr:.5f} | "
        f"train_acc={train_acc:.4f} train_loss={train_loss:.4f} | "
        f"val_acc={val_metrics['acc']:.4f} val_loss={val_metrics['loss']:.4f} | "
        f"best_val_acc={best_val_acc:.4f} @ epoch {best_epoch:03d} | "
        f"time={epoch_time:.1f}s"
    )

    scheduler.step()

if best_state_dict is None:
    raise RuntimeError("Training failed: no best checkpoint was stored.")

model.load_state_dict(best_state_dict)

final_train_metrics = evaluate(train_loader)
final_val_metrics = evaluate(val_loader)
final_test_metrics = evaluate(test_loader)

print("\nFinal evaluation using best validation checkpoint:")
print(
    f"best_epoch={best_epoch:03d} | "
    f"train_acc={final_train_metrics['acc']:.4f} train_loss={final_train_metrics['loss']:.4f} | "
    f"val_acc={final_val_metrics['acc']:.4f} val_loss={final_val_metrics['loss']:.4f} | "
    f"test_acc={final_test_metrics['acc']:.4f} test_loss={final_test_metrics['loss']:.4f}"
)

c:\Users\linki\anaconda3\envs\auto-ml\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train batches: 98
Validation batches: 98
Test batches: 40
Epoch 001/200 | lr=0.10000 | train_acc=0.2866 train_loss=2.1654 | val_acc=0.3067 val_loss=2.1459 | best_val_acc=0.3067 @ epoch 001 | time=26.6s
Epoch 002/200 | lr=0.09999 | train_acc=0.3676 train_loss=2.0865 | val_acc=0.3750 val_loss=2.0785 | best_val_acc=0.3750 @ epoch 002 | time=25.8s
Epoch 003/200 | lr=0.09998 | train_acc=0.4100 train_loss=2.0442 | val_acc=0.3964 val_loss=2.0574 | best_val_acc=0.3964 @ epoch 003 | time=25.4s
Epoch 004/200 | lr=0.09994 | train_acc=0.4360 train_loss=2.0193 | val_acc=0.4490 val_loss=2.0063 | best_val_acc=0.4490 @ epoch 004 | time=25.1s
Epoch 005/200 | lr=0.09990 | train_acc=0.4578 train_loss=1.9995 | val_acc=0.4232 val_loss=2.0311 | best_val_acc=0.4490 @ epoch 004 | time=25.3s
Epoch 006/200 | lr=0.09985 | train_acc=0.4778 train_loss=1.9787 | val_acc=0.4603 val_loss=1.9959 | best_val_acc=0.4603 @ epoch 006 | time=26.0s
Epoch 007/200 | lr=0.09978 | train_acc=0.5038 train_loss=1.9527 | val_acc=0.48

In [24]:
# Count model parameters
model = nn_core.build_model_from_dict_via_pydantic(model_spec).to(torch_device)

# all parameters in the model
pytorch_total_params = sum(p.numel() for p in model.parameters())

# trainable parameters in the model
pytorch_total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {pytorch_total_params}")
print(f"Trainable parameters: {pytorch_total_trainable_params}")


Total parameters: 1976650
Trainable parameters: 1976650


Get session model stats

In [65]:
session_models = get_session_models(benchmarking_session_id)
print(f"Session {benchmarking_session_id} has {len(session_models)} models.")

Session 16368e2889ea1d11e0bf89018bbdcd022ca5da018e647f40d823379ce0907fdf has 9 models.


In [29]:
session_training_progress = get_session_training_progress(benchmarking_session_id)

# Training accuracy and running max
training_accuracies = session_training_progress['accuracy_score'].dropna()
running_max_accuracy = training_accuracies.cummax()

window_start_5 = training_accuracies.rolling(window=5, min_periods=1).apply(lambda s: s.iloc[0], raw=False)
window_gain_5 = training_accuracies - window_start_5
window_gain_5 = window_gain_5.clip(lower=0)

df = pd.DataFrame({
    "call": training_accuracies.index + 1,
    "training_accuracy": training_accuracies.values,
    "running_max_accuracy": running_max_accuracy.values,
    "delta_to_max": running_max_accuracy.values - training_accuracies.values,
    "gain_5_calls_window": window_gain_5.values,
})

print(df)

#print("(" + df["call"].astype(str) + ", " +df["running_max_accuracy"].round(4).astype(str) + ")")

    call  training_accuracy  running_max_accuracy  delta_to_max  \
0      1              56.57                 56.57          0.00   
1      2              70.18                 70.18          0.00   
2      3              69.59                 70.18          0.59   
3      4              74.47                 74.47          0.00   
4      5              79.15                 79.15          0.00   
5      6              82.11                 82.11          0.00   
6      7              86.26                 86.26          0.00   
7      8              87.73                 87.73          0.00   
8      9              87.51                 87.73          0.22   
9     10              88.72                 88.72          0.00   
10    11              88.73                 88.73          0.00   
11    12              81.19                 88.73          7.54   

    gain_5_calls_window  
0                  0.00  
1                 13.61  
2                 13.02  
3                 17.90 

In [22]:
# Count average tokens in model specs for the session
encoding = tiktoken.encoding_for_model("gpt-5-")

session_model_tokens = pd.DataFrame(columns=["model_id", "num_tokens"])

for index, row in session_models.iterrows():
    model_id = row["id"]
    model_spec = json.dumps(row["nn_taxonomy"])
    num_tokens = len(encoding.encode(model_spec))
    session_model_tokens = pd.concat([session_model_tokens, pd.DataFrame({"model_id": [model_id], "num_tokens": [num_tokens]})], ignore_index=True)

print(f"Average tokens in model specs for session {benchmarking_session_id}: {session_model_tokens['num_tokens'].mean():.2f}")
print("\n")
print(session_model_tokens)


Average tokens in model specs for session 16368e2889ea1d11e0bf89018bbdcd022ca5da018e647f40d823379ce0907fdf: 1191.78


                               model_id num_tokens
0  55bf354e-e2cd-4bab-8e80-fd637548bff8       1687
1  d256ef6b-4447-4e01-b2ff-a52367ad5d39       1471
2  6b2c31ea-4ace-432f-b89c-0fceee27f5a3       1019
3  548372c6-1b4e-41cc-a49c-ec3ccdf39937        897
4  560187f2-c599-495a-a7c0-ea206e9e19e5       1602
5  e99bb7ce-e250-4c2f-bdbf-22ee8ff48aa7       1378
6  ea8cf96d-9cf2-4c7c-9c74-a04368afbb26        928
7  aaebefc2-6405-415d-b429-b67046aa1ee4        997
8  b5a50c7b-f4bf-4bd1-891d-bd78807aabae        747


Analyze model uniqueness

In [66]:
from collections import Counter
from itertools import combinations

from IPython.display import display

import nn_core

allowed_layers = {
    layer_name.removesuffix("Config"): params
    for layer_name, params in nn_core.get_allowed_layers().items()
}
allowed_layer_types = set(nn_core.LAYER_REGISTRY.keys())
missing_helper_layers = sorted(allowed_layer_types - set(allowed_layers.keys()))
if missing_helper_layers:
    print(
        "nn_core.get_allowed_layers() omits canonical registry entries; using LAYER_REGISTRY as the allowlist for: "
        + ", ".join(missing_helper_layers)
    )

models_to_compare = session_models.copy()

def _to_python(value):
    if isinstance(value, dict):
        return {key: _to_python(val) for key, val in sorted(value.items())}
    if isinstance(value, list):
        return [_to_python(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_python(item) for item in value)
    return value


def _as_spec(raw_spec):
    if isinstance(raw_spec, str):
        return json.loads(raw_spec)
    return raw_spec


def _layer_family(layer_type):
    if layer_type.startswith("Conv"):
        return "Conv"
    if "Pool" in layer_type:
        return "Pool"
    if "Norm" in layer_type:
        return "Norm"
    if layer_type in {"Linear"}:
        return "Dense"
    if layer_type.startswith("Dropout"):
        return "Dropout"
    if layer_type in {"Flatten", "Unflatten"}:
        return "Reshape"
    if layer_type in {"ReLU", "LeakyReLU", "Tanh", "Softmax", "Sigmoid", "GELU", "SiLU", "Mish"}:
        return "Activation"
    return "Other"


def _layer_signature(layer_dict):
    layer_type = layer_dict.get("layer_type")
    if layer_type not in allowed_layer_types:
        raise ValueError(f"Unsupported layer_type '{layer_type}'. Allowed types: {sorted(allowed_layer_types)}")
    params = {key: _to_python(value) for key, value in layer_dict.items() if key not in {"name", "layer_type"}}
    return f"{layer_type}|" + json.dumps(params, sort_keys=True, separators=(",", ":"), default=str)


def _weighted_jaccard(counter_a, counter_b):
    keys = set(counter_a) | set(counter_b)
    if not keys:
        return 1.0
    numerator = sum(min(counter_a.get(key, 0), counter_b.get(key, 0)) for key in keys)
    denominator = sum(max(counter_a.get(key, 0), counter_b.get(key, 0)) for key in keys)
    return numerator / denominator if denominator else 1.0


analysis_rows = []
layer_type_columns = []
layer_signature_columns = []
edge_motif_columns = []
exact_edge_columns = []

for _, row in models_to_compare.iterrows():
    model_id = str(row["id"])
    model_name = row.get("name") or model_id[:8]
    spec = _as_spec(row["nn_taxonomy"])

    validation = nn_core.validate_architecture_dict(spec)
    if not validation.valid:
        raise ValueError(
            f"Model {model_id} ({model_name}) failed nn_core validation:\n"
            + "\n".join(f"- {error}" for error in validation.errors)
        )

    nodes = spec["nodes"]
    edges = spec.get("edges", [])

    node_layer_type_counts = Counter()
    node_family_counts = Counter()
    node_signature_counts = Counter()
    node_signature_by_id = {}

    for node_id, node in nodes.items():
        layer = node["layer"]
        layer_type = layer["layer_type"]
        signature = _layer_signature(layer)
        node_layer_type_counts[layer_type] += 1
        node_family_counts[_layer_family(layer_type)] += 1
        node_signature_counts[signature] += 1
        node_signature_by_id[node_id] = signature

    edge_motif_counts = Counter()
    exact_edge_counts = Counter()
    skip_edge_count = 0

    for edge in edges:
        source_signature = node_signature_by_id[edge["source"]]
        target_signature = node_signature_by_id[edge["target"]]
        source_layer_type = nodes[edge["source"]]["layer"]["layer_type"]
        target_layer_type = nodes[edge["target"]]["layer"]["layer_type"]
        edge_type = edge.get("edge_type", "sequential")

        if edge_type == "skip":
            skip_edge_count += 1

        exact_edge_counts[(source_signature, target_signature, edge_type)] += 1
        edge_motif_counts[(source_layer_type, target_layer_type, edge_type)] += 1

    analysis_rows.append(
        {
            "model_id": model_id,
            "model_name": model_name,
            "num_nodes": len(nodes),
            "num_edges": len(edges),
            "num_skip_edges": skip_edge_count,
            "num_layer_types": len(node_layer_type_counts),
            "num_layer_signatures": len(node_signature_counts),
            "num_edge_motifs": len(edge_motif_counts),
            "num_exact_edge_signatures": len(exact_edge_counts),
        }
    )

    for key, value in node_layer_type_counts.items():
        layer_type_columns.append({"model_name": model_name, "layer_type": key, "count": value})

    for key, value in node_signature_counts.items():
        layer_signature_columns.append(
            {
                "model_name": model_name,
                "layer_signature": key,
                "layer_type": key.split("|", 1)[0],
                "family": _layer_family(key.split("|", 1)[0]),
                "count": value,
            }
        )

    for key, value in edge_motif_counts.items():
        source_layer_type, target_layer_type, edge_type = key
        edge_motif_columns.append(
            {
                "model_name": model_name,
                "edge_motif": f"{source_layer_type} -> {target_layer_type} [{edge_type}]",
                "source_layer_type": source_layer_type,
                "target_layer_type": target_layer_type,
                "edge_type": edge_type,
                "count": value,
            }
        )

    for key, value in exact_edge_counts.items():
        source_signature, target_signature, edge_type = key
        exact_edge_columns.append(
            {
                "model_name": model_name,
                "exact_edge_signature": f"{source_signature} -> {target_signature} [{edge_type}]",
                "edge_type": edge_type,
                "count": value,
            }
        )

summary_df = pd.DataFrame(analysis_rows)
layer_type_counts_df = (
    pd.DataFrame(layer_type_columns)
    .pivot_table(index="layer_type", columns="model_name", values="count", aggfunc="sum", fill_value=0)
    .sort_index()
)
layer_signature_counts_df = (
    pd.DataFrame(layer_signature_columns)
    .groupby(["layer_signature", "layer_type", "family"], as_index=False)["count"]
    .sum()
    .sort_values(["count", "layer_type"], ascending=[False, True])
)
edge_motif_counts_df = (
    pd.DataFrame(edge_motif_columns)
    .groupby(["edge_motif", "source_layer_type", "target_layer_type", "edge_type"], as_index=False)["count"]
    .sum()
    .sort_values(["count", "edge_motif"], ascending=[False, True])
)
exact_edge_counts_df = (
    pd.DataFrame(exact_edge_columns)
    .groupby(["exact_edge_signature", "edge_type"], as_index=False)["count"]
    .sum()
    .sort_values(["count", "exact_edge_signature"], ascending=[False, True])
)

pairwise_rows = []
model_feature_cache = {}
for _, row in models_to_compare.iterrows():
    model_id = str(row["id"])
    model_name = row.get("name") or model_id[:8]
    spec = _as_spec(row["nn_taxonomy"])
    nodes = spec["nodes"]
    edges = spec.get("edges", [])

    node_layer_type_counts = Counter()
    node_signature_counts = Counter()
    node_signature_by_id = {}
    edge_motif_counts = Counter()

    for node_id, node in nodes.items():
        layer = node["layer"]
        layer_type = layer["layer_type"]
        signature = _layer_signature(layer)
        node_layer_type_counts[layer_type] += 1
        node_signature_counts[signature] += 1
        node_signature_by_id[node_id] = signature

    for edge in edges:
        source_layer_type = nodes[edge["source"]]["layer"]["layer_type"]
        target_layer_type = nodes[edge["target"]]["layer"]["layer_type"]
        edge_type = edge.get("edge_type", "sequential")
        edge_motif_counts[(source_layer_type, target_layer_type, edge_type)] += 1

    model_feature_cache[model_name] = {
        "layer_type_counts": node_layer_type_counts,
        "layer_signature_counts": node_signature_counts,
        "edge_motif_counts": edge_motif_counts,
    }

for model_a, model_b in combinations(model_feature_cache.keys(), 2):
    features_a = model_feature_cache[model_a]
    features_b = model_feature_cache[model_b]

    layer_type_similarity = _weighted_jaccard(features_a["layer_type_counts"], features_b["layer_type_counts"])
    layer_signature_similarity = _weighted_jaccard(features_a["layer_signature_counts"], features_b["layer_signature_counts"])
    edge_motif_similarity = _weighted_jaccard(features_a["edge_motif_counts"], features_b["edge_motif_counts"])
    overall_similarity = (layer_signature_similarity + edge_motif_similarity) / 2

    pairwise_rows.append(
        {
            "model_a": model_a,
            "model_b": model_b,
            "layer_type_similarity": layer_type_similarity,
            "layer_signature_similarity": layer_signature_similarity,
            "edge_motif_similarity": edge_motif_similarity,
            "overall_similarity": overall_similarity,
            "overall_difference": 1 - overall_similarity,
        }
    )

pairwise_df = pd.DataFrame(pairwise_rows).sort_values(
    ["overall_difference", "model_a", "model_b"], ascending=[False, True, True]
)

print("Validated and compared models using nn_core canonical layer rules.")
print(f"Allowed layer types ({len(allowed_layer_types)}): {sorted(allowed_layer_types)}")
print()
print("Per-model summary:")
display(summary_df)
print("Layer type counts:")
display(layer_type_counts_df)
print("Top parameterized layer signatures:")
display(layer_signature_counts_df.head(25))
print("Top edge motifs:")
display(edge_motif_counts_df.head(25))
print("Exact edge signatures:")
display(exact_edge_counts_df.head(25))
print("Pairwise similarity / difference scores:")
display(pairwise_df)


nn_core.get_allowed_layers() omits canonical registry entries; using LAYER_REGISTRY as the allowlist for: AdaptiveAvgPool2d
Validated and compared models using nn_core canonical layer rules.
Allowed layer types (26): ['AdaptiveAvgPool2d', 'AvgPool2d', 'BatchNorm1d', 'BatchNorm2d', 'Conv1d', 'Conv2d', 'ConvTranspose1d', 'ConvTranspose2d', 'Dropout', 'Dropout2d', 'Flatten', 'GELU', 'GroupNorm', 'InstanceNorm2d', 'LayerNorm', 'LeakyReLU', 'Linear', 'MaxPool1d', 'MaxPool2d', 'Mish', 'ReLU', 'SiLU', 'Sigmoid', 'Softmax', 'Tanh', 'Unflatten']

Per-model summary:


,model_id,model_name,num_nodes,num_edges,num_skip_edges,num_layer_types,num_layer_signatures,num_edge_motifs,num_exact_edge_signatures
0,b5a50c7b-f4bf-4bd1-891d-bd78807aabae,CompactResidualStyleCNN_32x32_10way,13,12,0,7,10,9,11
1,aaebefc2-6405-415d-b429-b67046aa1ee4,GradientStableLeakyCNN_CIFAR10,16,15,0,7,11,10,14
2,ea8cf96d-9cf2-4c7c-9c74-a04368afbb26,SimpleResNetV2StyleLeakyCNN_32x32,14,15,2,6,10,10,15
3,e99bb7ce-e250-4c2f-bdbf-22ee8ff48aa7,WideResidualBNConvNet_32x32,22,23,2,8,15,11,21
4,560187f2-c599-495a-a7c0-ea206e9e19e5,LightweightResidualDepthwiseCNN_32x32,24,24,1,8,17,12,22
5,548372c6-1b4e-41cc-a49c-ec3ccdf39937,ResidualWideCNN_CIFAR10,14,14,1,6,10,9,14
6,6b2c31ea-4ace-432f-b89c-0fceee27f5a3,PreActResidualCNN_CIFAR10,16,17,2,7,12,11,16
7,d256ef6b-4447-4e01-b2ff-a52367ad5d39,DeepPreActResidualCNN_CIFAR10,23,25,3,8,16,12,23
8,55bf354e-e2cd-4bab-8e80-fd637548bff8,RefinedPreActResidualCNN_CIFAR10,27,29,3,8,19,14,27


Layer type counts:


model_name,CompactResidualStyleCNN_32x32_10way,DeepPreActResidualCNN_CIFAR10,GradientStableLeakyCNN_CIFAR10,LightweightResidualDepthwiseCNN_32x32,PreActResidualCNN_CIFAR10,RefinedPreActResidualCNN_CIFAR10,ResidualWideCNN_CIFAR10,SimpleResNetV2StyleLeakyCNN_32x32,WideResidualBNConvNet_32x32
layer_type,,,,,,,,,
BatchNorm2d,2,5,0,5,3,5,0,0,5
Conv2d,2,6,4,7,4,6,4,4,5
Dropout,0,1,1,1,0,3,0,0,1
Flatten,1,1,1,1,1,1,1,1,1
LeakyReLU,0,0,5,6,0,0,5,5,0
Linear,2,2,2,2,2,3,2,2,2
MaxPool2d,2,1,2,1,1,1,1,1,1
ReLU,3,6,0,0,4,7,0,0,6
Softmax,1,1,1,1,1,1,1,1,1


Top parameterized layer signatures:


...

,edge_motif,source_layer_type,target_layer_type,edge_type,count
2,Conv2d -> BatchNorm2d [sequential],Conv2d,BatchNorm2d,sequential,25
1,BatchNorm2d -> ReLU [sequential],BatchNorm2d,ReLU,sequential,20
20,ReLU -> Conv2d [sequential],ReLU,Conv2d,sequential,15
4,Conv2d -> LeakyReLU [sequential],Conv2d,LeakyReLU,sequential,12
9,LeakyReLU -> Conv2d [sequential],LeakyReLU,Conv2d,sequential,12
8,Flatten -> Linear [sequential],Flatten,Linear,sequential,9
17,Linear -> Softmax [sequential],Linear,Softmax,sequential,9
19,MaxPool2d -> Flatten [sequential],MaxPool2d,Flatten,sequential,9
21,ReLU -> Conv2d [skip],ReLU,Conv2d,skip,7
16,Linear -> ReLU [sequential],Linear,ReLU,sequential,6


Exact edge signatures:


,exact_edge_signature,edge_type,count
56,"Linear|{""bias"":true,""in_features"":""auto"",""out_...",sequential,9
65,"MaxPool2d|{""kernel_size"":2,""stride"":2} -> Flat...",sequential,9
5,"BatchNorm2d|{""affine"":true,""num_features"":64} ...",sequential,8
1,"BatchNorm2d|{""affine"":true,""num_features"":128}...",sequential,7
54,"LeakyReLU|{""negative_slope"":0.01} -> MaxPool2d...",sequential,5
15,"Conv2d|{""in_channels"":128,""kernel_size"":3,""out...",sequential,4
26,"Conv2d|{""in_channels"":3,""kernel_size"":3,""out_c...",sequential,4
31,"Conv2d|{""in_channels"":64,""kernel_size"":3,""out_...",sequential,4
39,"Flatten|{} -> Linear|{""in_features"":""auto"",""ou...",sequential,4
50,"LeakyReLU|{""negative_slope"":0.01} -> Conv2d|{""...",sequential,4


Pairwise similarity / difference scores:


,model_a,model_b,layer_type_similarity,layer_signature_similarity,edge_motif_similarity,overall_similarity,overall_difference
15,SimpleResNetV2StyleLeakyCNN_32x32,WideResidualBNConvNet_32x32,0.333333,0.125000,0.085714,0.105357,0.894643
19,SimpleResNetV2StyleLeakyCNN_32x32,DeepPreActResidualCNN_CIFAR10,0.321429,0.156250,0.081081,0.118666,0.881334
20,SimpleResNetV2StyleLeakyCNN_32x32,RefinedPreActResidualCNN_CIFAR10,0.281250,0.171429,0.073171,0.122300,0.877700
14,GradientStableLeakyCNN_CIFAR10,RefinedPreActResidualCNN_CIFAR10,0.303030,0.162162,0.100000,0.131081,0.868919
13,GradientStableLeakyCNN_CIFAR10,DeepPreActResidualCNN_CIFAR10,0.344828,0.181818,0.111111,0.146465,0.853535
18,SimpleResNetV2StyleLeakyCNN_32x32,PreActResidualCNN_CIFAR10,0.428571,0.200000,0.103448,0.151724,0.848276
4,CompactResidualStyleCNN_32x32_10way,ResidualWideCNN_CIFAR10,0.350000,0.173913,0.130435,0.152174,0.847826
9,GradientStableLeakyCNN_CIFAR10,WideResidualBNConvNet_32x32,0.357143,0.187500,0.117647,0.152574,0.847426
32,ResidualWideCNN_CIFAR10,RefinedPreActResidualCNN_CIFAR10,0.281250,0.242424,0.075000,0.158712,0.841288
3,CompactResidualStyleCNN_32x32_10way,LightweightResidualDepthwiseCNN_32x32,0.321429,0.156250,0.161290,0.158770,0.841230


In [76]:
summary_order_df = summary_df.reset_index(drop=True).reset_index().rename(columns={"index": "generation_index"})

pairwise_indexed = pairwise_df.merge(
    summary_order_df[["model_name", "generation_index"]].rename(columns={"model_name": "model_a"}),
    on="model_a",
    how="left",
)
pairwise_indexed = pairwise_indexed.merge(
    summary_order_df[["model_name", "generation_index"]].rename(columns={"model_name": "model_b", "generation_index": "peer_generation_index"}),
    on="model_b",
    how="left",
)

pairwise_symmetric = pd.concat(
    [
        pairwise_indexed[["generation_index", "peer_generation_index", "overall_similarity", "overall_difference"]],
        pairwise_indexed[["peer_generation_index", "generation_index", "overall_similarity", "overall_difference"]]
        .rename(columns={"peer_generation_index": "generation_index", "generation_index": "peer_generation_index"}),
    ],
    ignore_index=True,
)

nearest_neighbor_df = (
    pairwise_symmetric.groupby("generation_index", as_index=False)
    .agg(
        nearest_neighbor_similarity=("overall_similarity", "max"),
        nearest_neighbor_difference=("overall_difference", "min"),
    )
)

# Normalize merge keys (UUID vs str) and keep the latest accuracy per model_id.
summary_order_df_merge = summary_order_df.copy()
summary_order_df_merge["model_id"] = summary_order_df_merge["model_id"].astype(str)

session_training_results_merge = session_training_progress[["model_id", "accuracy_score", "created_at"]].copy()
session_training_results_merge["model_id"] = session_training_results_merge["model_id"].astype(str)
session_training_results_merge = (
    session_training_results_merge.sort_values("created_at")
    .drop_duplicates(subset=["model_id"], keep="last")
    [["model_id", "accuracy_score"]]
)

summary_order_df_merged = summary_order_df_merge.merge(session_training_results_merge, how="left", on="model_id")

paper_table_df = (
    summary_order_df_merged[["generation_index", "model_name", "num_nodes", "num_edges", "num_layer_signatures", "accuracy_score"]]
    .merge(nearest_neighbor_df[["generation_index", "nearest_neighbor_similarity"]], on="generation_index", how="left")
    .sort_values("generation_index")
    .reset_index(drop=True)
)
paper_table_df["nearest_neighbor_similarity"] = paper_table_df["nearest_neighbor_similarity"].round(3)
paper_table_df = paper_table_df[["model_name", "num_nodes", "num_edges", "num_layer_signatures", "accuracy_score"]]

print("Paper table (5 columns):")
print(f"Matched training accuracies for {paper_table_df['accuracy_score'].notna().sum()} / {len(paper_table_df)} models")
display(paper_table_df)

Paper table (5 columns):
Matched training accuracies for 3 / 9 models


,model_name,num_nodes,num_edges,num_layer_signatures,accuracy_score
0,CompactResidualStyleCNN_32x32_10way,13,12,10,56.57
1,GradientStableLeakyCNN_CIFAR10,16,15,11,NaN
2,SimpleResNetV2StyleLeakyCNN_32x32,14,15,10,NaN
3,WideResidualBNConvNet_32x32,22,23,15,81.19
4,LightweightResidualDepthwiseCNN_32x32,24,24,17,69.59
5,ResidualWideCNN_CIFAR10,14,14,10,NaN
6,PreActResidualCNN_CIFAR10,16,17,12,NaN
7,DeepPreActResidualCNN_CIFAR10,23,25,16,NaN
8,RefinedPreActResidualCNN_CIFAR10,27,29,19,NaN


Check architecture validations

In [30]:
with open("benchmarking_facilitator.log", "r") as f:
    log_contents = f.read()

In [33]:
validation_valid = log_contents.count("POST /validate HTTP/1.1\" 200")
validation_invalid = log_contents.count("POST /validate HTTP/1.1\" 400")

print(f"Validation requests: {validation_valid} valid, {validation_invalid} invalid")

Validation requests: 12 valid, 2 invalid
